## SpottingSalmon 
Image extraction script.
- To extract video frames as images for use in model training. 

#### Set up:

In [0]:

%pip install opencv-python numpy==1.26.4 --force-reinstall

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import subprocess
import json
import cv2
from PIL import Image

#### Step 1: Check the metadata

In [0]:
# File path to one video for QC
video_file = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"

# Meta data from video
cmd = [
    "ffprobe",
    "-v", "error",
    "-select_streams", "v:0",
    "-show_entries", "stream=codec_name,width,height,r_frame_rate",
    "-of", "json",
    video_file
]


result = subprocess.run(cmd, capture_output=True, text=True)
info = json.loads(result.stdout)

video = info["streams"][0]
codec = video["codec_name"]
width = video["width"]
height = video["height"]
num, denom = map(int, video["r_frame_rate"].split("/"))
fps = round(num / denom, 2)

# Show relevant metadata
print(f"Codec: {codec}")
print(f"Resolution: {width}x{height}")
print(f"FPS: {fps}")



#### Step 2: Extract each video frame as a .jpeg

In [0]:

# Set the directory where the video files are located
input_dir = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/" 

# Set the directory where the extracted frames will be saved
output_dir = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/extracted_frames"
os.makedirs(output_dir, exist_ok=True)

# Supported video file extensions
video_extensions = ('.mp4')

# Function to extract frames from a single video
def extract_frames(video_path, output_folder, frame_interval=1):
    os.makedirs(output_folder, exist_ok=True) # Create the output folder
    cap = cv2.VideoCapture(video_path) # Open the video file

    if not cap.isOpened(): # Check if the video file was opened successfully
        print(f"Error: Could not open video file {video_path}")
        return

    frame_count = 0 # Counter for total frames
    saved_count = 0 # Counter for saved frames

    while True: # Loop through each frame
        ret, frame = cap.read() # Read the next frame
        if not ret: 
            break # Break the loop if there are no more frames

        if frame_count % frame_interval == 0: # Save the frame if it's a multiple of the frame interval
            frame_filename = os.path.join(output_folder, f"frame{frame_count:05d}.jpg")
            cv2.imwrite(frame_filename, frame) # Save the frame as img
            saved_count += 1 

        frame_count += 1

    cap.release()
    print(f"Extracted {saved_count} frames from '{os.path.basename(video_path)}'")

# Loop through each file in the input directory
for filename in os.listdir(input_dir):
    if filename.lower().endswith(video_extensions):
        video_path = os.path.join(input_dir, filename)
        video_name = os.path.splitext(filename)[0]
        video_output_folder = os.path.join(output_dir, video_name)

        extract_frames(video_path, video_output_folder, frame_interval=1)




#### Step 3: Inspect data

In [0]:
# Directory for 1 video for QC
frame_dir = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/extracted_frames/4_2021-07-06_14-11-37x/"

# Show only the first 5 frames
for i, filename in enumerate(sorted(os.listdir(frame_dir))):
    if filename.startswith("frame") and filename.endswith(".jpg"):
        if i >= 5:  # Limit to first 5
            break
        frame_path = os.path.join(frame_dir, filename)
        img = Image.open(frame_path)
        display(img)
